<a href="https://colab.research.google.com/github/rishabhraj588/AI-Assistant/blob/main/ml_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import joblib


In [3]:
df=pd.read_csv("/content/synthetic_fraud_10000.csv")

In [4]:
df.head()

,amount,merchant,user_id,time,location,device,is_fraud
0,175.36,StoreC,U0796,2026-04-16 08:51:47,Kolkata,Mobile,0
1,390.00,StoreL,U1691,2026-04-17 00:44:23,Kolkata,Mobile,0
2,315.71,StoreR,U2226,2026-04-16 01:15:52,Hyderabad,Web,0
3,27.44,StoreG,U0385,2026-04-21 09:05:47,Bangalore,Web,0
4,292.79,StoreS,U0644,2026-04-13 06:27:55,Hyderabad,POS,0


In [5]:
df.tail()



,amount,merchant,user_id,time,location,device,is_fraud
9995,341.79,StoreB,U3727,2026-04-09 22:31:59,Chennai,Web,0
9996,353.40,StoreB,U6415,2026-04-21 21:29:36,Hyderabad,Mobile,0
9997,78.33,StoreA,U6776,2026-04-07 07:11:52,Kolkata,Mobile,0
9998,390.01,StoreA,U7700,2026-04-27 05:07:26,Mumbai,Mobile,0
9999,167.15,StoreX,U9820,2026-04-08 19:03:46,Chennai,Mobile,0


In [6]:
#checking the missing values
df.isnull().sum()

,0
amount,0
merchant,0
user_id,0
time,0
location,0
device,0
is_fraud,0


In [7]:
#distribution of legit transaction and fraudalant transaction
df['is_fraud'].value_counts()

,count
is_fraud,
0,9800
1,200


In [8]:
legit=df[df.is_fraud==0]
fraud=df[df.is_fraud==1]

In [9]:
legit.shape
fraud.shape

(200, 7)

In [10]:
fraud.amount.describe()

,amount
count,200.000000
mean,2785.789800
std,1340.269892
min,522.300000
25%,1575.585000
50%,2896.695000
75%,3990.760000
max,4984.830000


In [11]:
df.groupby('is_fraud')['amount'].mean()

,amount
is_fraud,
0,252.048049
1,2785.789800


In [12]:
legit_sample=legit.sample(n=200)

In [13]:
new_dataset = pd.concat([legit_sample, fraud], axis=0)

In [14]:
new_dataset['is_fraud'].value_counts()

,count
is_fraud,
0,200
1,200


In [15]:
new_dataset.groupby('is_fraud').mean()

TypeError: agg function failed [how->mean,dtype->object]

In [16]:
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

In [17]:
X

,amount,merchant,user_id,time,location,device
0,175.36,StoreC,U0796,2026-04-16 08:51:47,Kolkata,Mobile
1,390.00,StoreL,U1691,2026-04-17 00:44:23,Kolkata,Mobile
2,315.71,StoreR,U2226,2026-04-16 01:15:52,Hyderabad,Web
3,27.44,StoreG,U0385,2026-04-21 09:05:47,Bangalore,Web
4,292.79,StoreS,U0644,2026-04-13 06:27:55,Hyderabad,POS
...,...,...,...,...,...,...
9995,341.79,StoreB,U3727,2026-04-09 22:31:59,Chennai,Web
9996,353.40,StoreB,U6415,2026-04-21 21:29:36,Hyderabad,Mobile
9997,78.33,StoreA,U6776,2026-04-07 07:11:52,Kolkata,Mobile
9998,390.01,StoreA,U7700,2026-04-27 05:07:26,Mumbai,Mobile


In [18]:
y

,is_fraud
0,0
1,0
2,0
3,0
4,0
...,...
9995,0
9996,0
9997,0
9998,0


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=2)

In [20]:
print(X.shape, X_train.shape, X_test.shape)

(10000, 6) (8000, 6) (2000, 6)


In [21]:
model=LogisticRegression()

In [22]:
scaler = StandardScaler()

# Preprocessing steps for X_train and X_test before scaling

# 1. Drop 'user_id' column as it's an identifier and not directly useful for prediction
X_train_processed = X_train.drop('user_id', axis=1)
X_test_processed = X_test.drop('user_id', axis=1)

# 2. Convert 'time' column to numerical Unix timestamp
X_train_processed['time'] = pd.to_datetime(X_train_processed['time']).astype(int) / 10**9
X_test_processed['time'] = pd.to_datetime(X_test_processed['time']).astype(int) / 10**9

# 3. One-hot encode categorical columns: 'merchant', 'location', 'device'
categorical_cols = ['merchant', 'location', 'device']

X_train_processed = pd.get_dummies(X_train_processed, columns=categorical_cols, drop_first=True)
X_test_processed = pd.get_dummies(X_test_processed, columns=categorical_cols, drop_first=True)

# Align columns to ensure both train and test sets have the same columns in the same order
# This handles cases where certain categories might be present in one set but not the other
train_cols = X_train_processed.columns
test_cols = X_test_processed.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    X_test_processed[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X_train_processed[c] = 0

X_test_processed = X_test_processed[train_cols] # Reorder test columns to match train columns

# Now, apply StandardScaler to the processed numerical data
X_train_scaled = scaler.fit_transform(X_train_processed)
X_test_scaled = scaler.transform(X_test_processed)

In [23]:
model.fit(X_train_scaled, y_train)

LogisticRegression()

In [24]:
X_train_prediction = model.predict(X_train_scaled)
training_data_accuracy = accuracy_score(X_train_prediction, y_train)
print('Accuracy on Training data : ', training_data_accuracy)

Accuracy on Training data :  0.999


In [25]:
X_test_prediction = model.predict(X_test_scaled)
test_data_accuracy = accuracy_score(X_test_prediction, y_test)
print('Accuracy score on Test Data : ', test_data_accuracy)

Accuracy score on Test Data :  0.997


In [26]:
preds = model.predict(X_test_scaled)
print("Classification Report:\n", classification_report(y_test, preds))

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1960
           1       1.00      0.85      0.92        40

    accuracy                           1.00      2000
   macro avg       1.00      0.93      0.96      2000
weighted avg       1.00      1.00      1.00      2000



In [27]:
joblib.dump(model, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [28]:
input_data =(360.15,'StoreF','U5906','2026-04-07 10:19:23','Bangalore','Mobile')

# Create a DataFrame for the input data, mirroring the original X structure
input_df = pd.DataFrame([input_data], columns=['amount', 'merchant', 'user_id', 'time', 'location', 'device'])

# Apply the same preprocessing steps as X_train and X_test
# 1. Drop 'user_id' column
input_processed = input_df.drop('user_id', axis=1)

# 2. Convert 'time' column to numerical Unix timestamp
input_processed['time'] = pd.to_datetime(input_processed['time']).astype(int) / 10**9

# 3. One-hot encode categorical columns
categorical_cols = ['merchant', 'location', 'device']
input_processed = pd.get_dummies(input_processed, columns=categorical_cols, drop_first=True)

# Align columns with the training data columns (X_train_processed)
# We need to ensure the input_processed DataFrame has the exact same columns as X_train_processed in the same order.
# X_train_processed.columns is available from the kernel state.

# Create a template DataFrame with all training columns and fill with 0s
# This assumes X_train_processed is accessible from previous execution
train_cols = X_train_processed.columns
final_input_df = pd.DataFrame(0, index=[0], columns=train_cols)

# Copy the values from input_processed to final_input_df
for col in input_processed.columns:
    if col in final_input_df.columns:
        final_input_df[col] = input_processed[col]

# Standardize the preprocessed input data
std_data = scaler.transform(final_input_df)

# Predict
prediction = model.predict(std_data)
print("Prediction:", prediction)

if prediction[0] == 0:
    print("Transaction is Legitimate")

else:
    print("Transaction is Fraudulent")

Prediction: [0]
Transaction is Legitimate
